In [ ]:
# CELL 1: RUN THIS FIRST
print("Initiating Environment Clean & Install...")
!pip install -qU trl transformers unsloth datasets
!pip uninstall -y wandb
print("\n[SYSTEM ALERT]: Installation complete.")
print("CRITICAL STEP: You MUST click 'Runtime' -> 'Restart session' at the top of the screen before running Cell 2.")

Initiating Environment Clean & Install...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.3/432.3 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 115.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 120.2 MB/s eta 0:00:00

In [ ]:
import os
import sys

# --- 1. AUTOMATED DEPENDENCY INSTALLATION ---
try:
    import unsloth
    import trl
    import datasets
    print("[SYSTEM LOG]: All libraries present.")
except ImportError:
    print("[SYSTEM LOG]: Missing dependencies detected. Initiating automated install...")
    os.system("pip uninstall -y wandb")
    os.system("pip install --no-deps trl transformers datasets")
    os.system("pip install unsloth")
    print("[SYSTEM LOG]: Installation complete. Restarting Python kernel...")
    os.execv(sys.executable, ['python'] + sys.argv)

# --- 2. CORE IMPORTS & WANDB SUPPRESSION ---
import re
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_DISABLED"] = "true"

# --- 3. GOOGLE DRIVE MOUNTING ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = '/content/drive/MyDrive/AI_Reasoning_Project'
    os.makedirs(base_dir, exist_ok=True)
    print("\n[SYSTEM LOG]: Google Drive mounted successfully.")
except ImportError:
    print("\n[SYSTEM WARNING]: Not running in Colab. Using local directory.")
    base_dir = './AI_Reasoning_Project'
    os.makedirs(base_dir, exist_ok=True)

# --- 4. CORE BRAIN INITIALIZATION ---
max_seq_length = 1024
lora_rank = 16

print("Loading base model in 4-bit precision...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# --- 5. THE TRAINING ENVIRONMENT (MATHEMATICS) ---
print("Downloading GSM8K Mathematics Dataset...")
dataset = load_dataset("openai/gsm8k", "main", split="train")

SYSTEM_PROMPT = """
Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

def format_data(example):
    answer = example["answer"].split("#### ")[-1]
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]}
        ],
        "ground_truth": answer
    }

dataset = dataset.map(format_data)

# --- 6. THE REWARD SYSTEM ---
def extract_xml_answer(text: str) -> str:
    try:
        answer = text.split("<answer>")[-1]
        return answer.split("</answer>")[0].strip()
    except:
        return ""

def correctness_reward(completions, ground_truth, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]
    return [1.0 if r == gt else 0.0 for r, gt in zip(extracted_responses, ground_truth)]

def format_reward(completions, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    pattern = r"<reasoning>.*?</reasoning>\\s*<answer>.*?</answer>"
    return [0.5 if re.search(pattern, r, re.DOTALL) else 0.0 for r in responses]

# --- 7. THE REINFORCEMENT LEARNING LOOP ---
print("Configuring GRPO Reinforcement Loop...")
checkpoint_dir = os.path.join(base_dir, "grpo_reasoning_checkpoints")

training_args = GRPOConfig(
    use_vllm=False,
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=256,
    max_completion_length=max_seq_length - 256,
    max_steps=300,
    output_dir=checkpoint_dir,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    report_to="none"
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[correctness_reward, format_reward],
    args=training_args,
    train_dataset=dataset,
)

# --- 8. RESUME AND TRAIN ---
print("\nInitiating Training Protocol...")
try:
    trainer.train(resume_from_checkpoint=True)
    print("[SYSTEM LOG]: Resumed training from checkpoint.")
except (ValueError, TypeError, KeyError):
    print("[SYSTEM LOG]: Starting fresh training loop.")
    trainer.train()

# --- 9. SAVE THE FINAL MODEL ---
final_model_dir = os.path.join(base_dir, "final_reasoning_model")
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)
print(f"\n[SYSTEM LOG]: Final weights stored at: {final_model_dir}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[SYSTEM LOG]: All libraries present.
Mounted at /content/drive

[SYSTEM LOG]: Google Drive mounted successfully.
Loading base model in 4-bit precision...
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / format_reward / mean,rewards / format_reward / std
51,0.000000,0.250000,0.500000,234.000000,127.000000,304.000000,0.000000,234.000000,127.000000,304.000000,0,0,0,0,0,0.000108,0.250000,0.500000,0.000000,0.000000
52,0.000000,0.000000,0.000000,426.750000,306.000000,494.000000,0.000000,426.750000,306.000000,494.000000,No Log,No Log,No Log,No Log,No Log,0.000268,0.000000,0.000000,0.000000,0.000000
53,0.000000,0.000000,0.000000,213.750000,202.000000,230.000000,0.000000,213.750000,202.000000,230.000000,No Log,No Log,No Log,No Log,No Log,0.000205,0.000000,0.000000,0.000000,0.000000
54,0.000000,0.000000,0.000000,206.250000,129.000000,247.000000,0.000000,206.250000,129.000000,247.000000,No Log,No Log,No Log,No Log,No Log,0.000053,0.000000,0.000000,0.000000,0.000000
55,0.000000,0.000000,0.000000,164.250000,103.000000,228.000000,0.000000,164.250000,103.000000,228.000000,No Log,No Log,No Log,No Log,No Log,0.000643,0.000000,0.000000,0.000000,0.000000
56,0.000000,0.000000,0.000000,245.500000,155.000000,317.000000,0.000000,245.500000,155.000000,317.000000,No Log,No Log,No Log,No Log,No Log,0.000097,0.000000,0.000000,0.000000,0.000000
57,0.000000,0.000000,0.000000,326.500000,283.000000,444.000000,0.000000,326.500000,283.000000,444.000000,No Log,No Log,No Log,No Log,No Log,0.000147,0.000000,0.000000,0.000000,0.000000
58,0.000000,0.000000,0.000000,302.500000,191.000000,419.000000,0.000000,302.500000,191.000000,419.000000,No Log,No Log,No Log,No Log,No Log,0.000034,0.000000,0.000000,0.000000,0.000000
59,0.000000,0.250000,0.500000,251.250000,152.000000,385.000000,0.000000,251.250000,152.000000,385.000000,No Log,No Log,No Log,No Log,No Log,0.000403,0.250000,0.500000,0.000000,0.000000
60,0.000000,0.000000,0.000000,129.750000,94.000000,171.000000,0.000000,129.750000,94.000000,171.000000,No Log,No Log,No Log,No Log,No Log,0.000091,0.000000,0.000000,0.000000,0.000000


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / format_reward / mean,rewards / format_reward / std
51,0.000000,0.250000,0.500000,234.000000,127.000000,304.000000,0.000000,234.000000,127.000000,304.000000,0,0,0,0,0,0.000108,0.250000,0.500000,0.000000,0.000000
52,0.000000,0.000000,0.000000,426.750000,306.000000,494.000000,0.000000,426.750000,306.000000,494.000000,No Log,No Log,No Log,No Log,No Log,0.000268,0.000000,0.000000,0.000000,0.000000
53,0.000000,0.000000,0.000000,213.750000,202.000000,230.000000,0.000000,213.750000,202.000000,230.000000,No Log,No Log,No Log,No Log,No Log,0.000205,0.000000,0.000000,0.000000,0.000000
54,0.000000,0.000000,0.000000,206.250000,129.000000,247.000000,0.000000,206.250000,129.000000,247.000000,No Log,No Log,No Log,No Log,No Log,0.000053,0.000000,0.000000,0.000000,0.000000
55,0.000000,0.000000,0.000000,164.250000,103.000000,228.000000,0.000000,164.250000,103.000000,228.000000,No Log,No Log,No Log,No Log,No Log,0.000643,0.000000,0.000000,0.000000,0.000000
56,0.000000,0.000000,0.000000,245.500000,155.000000,317.000000,0.000000,245.500000,155.000000,317.000000,No Log,No Log,No Log,No Log,No Log,0.000097,0.000000,0.000000,0.000000,0.000000
57,0.000000,0.000000,0.000000,326.500000,283.000000,444.000000,0.000000,326.500000,283.000000,444.000000,No Log,No Log,No Log,No Log,No Log,0.000147,0.000000,0.000000,0.000000,0.000000
58,0.000000,0.000000,0.000000,302.500000,191.000000,419.000000,0.000000,302.500000,191.000000,419.000000,No Log,No Log,No Log,No Log,No Log,0.000034,0.000000,0.000000,0.000000,0.000000
59,0.000000,0.250000,0.500000,251.250000,152.000000,385.000000,0.000000,251.250000,152.000000,385.000000,No Log,No Log,No Log,No Log,No Log,0.000403,0.250000,0.500000,0.000000,0.000000
60,0.000000,0.000000,0.000000,129.750000,94.000000,171.000000,0.000000,129.750000,94.000000,171.000000,No Log,No Log,No Log,No Log,No Log,0.000091,0.000000,0.000000,0.000000,0.000000


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


[SYSTEM LOG]: Resumed training from checkpoint.

[SYSTEM LOG]: Final weights stored at: /content/drive/MyDrive/AI_Reasoning_Project/final_reasoning_model
